In [1]:
import pandas as pd

train = pd.read_pickle("../data/processed/train.pkl")
validation = pd.read_pickle("../data/processed/validation.pkl")
test = pd.read_pickle("../data/processed/test.pkl")

In [2]:
train.shape, validation.shape, test.shape

((473436, 15), (118664, 15), (121160, 15))

In [3]:
validation["naive_prediction"] = validation["lag_1"]

In [4]:
validation[["StockCode", "Date", "Demand", "lag_1", "naive_prediction", "target"]].head(
    15
)

,StockCode,Date,Demand,lag_1,naive_prediction,target
1036,10125,2011-08-20,0,20.0,20.0,0.0
1037,10125,2011-08-21,0,0.0,0.0,0.0
1038,10125,2011-08-22,0,0.0,0.0,0.0
1039,10125,2011-08-23,0,0.0,0.0,25.0
1040,10125,2011-08-24,25,0.0,0.0,0.0
1041,10125,2011-08-25,0,25.0,25.0,0.0
1042,10125,2011-08-26,0,0.0,0.0,0.0
1043,10125,2011-08-27,0,0.0,0.0,0.0
1044,10125,2011-08-28,0,0.0,0.0,0.0
1045,10125,2011-08-29,0,0.0,0.0,0.0


In [6]:
from sklearn.metrics import mean_absolute_error

mae_naive = mean_absolute_error(validation["target"], validation["naive_prediction"])

mae_naive

11.302534888424459

In [7]:
from sklearn.metrics import mean_squared_error

rmse_naive = (
    mean_squared_error(validation["target"], validation["naive_prediction"]) ** 0.5
)

rmse_naive

49.119678713502225

In [8]:
zero_actual = validation["target"] == 0

zero_prediction_correct = (validation.loc[zero_actual, "naive_prediction"] == 0).mean()

zero_prediction_correct

np.float64(0.7388078031879823)

In [10]:
baseline_data = (
    pd.concat([train, validation, test], ignore_index=True)
    .sort_values(["StockCode", "Date"])
    .copy()
)

In [11]:
baseline_data["ma_7"] = baseline_data.groupby("StockCode")["Demand"].transform(
    lambda x: x.rolling(7).mean()
)

In [12]:
baseline_data[["StockCode", "Date", "Demand", "ma_7"]].head(15)

,StockCode,Date,Demand,ma_7
0,10002,2010-12-29,0,NaN
1,10002,2010-12-30,0,NaN
2,10002,2010-12-31,0,NaN
3,10002,2011-01-01,0,NaN
4,10002,2011-01-02,0,NaN
5,10002,2011-01-03,0,NaN
6,10002,2011-01-04,0,0.000000
7,10002,2011-01-05,12,1.714286
8,10002,2011-01-06,60,10.285714
9,10002,2011-01-07,1,10.428571


In [13]:
baseline_data[["StockCode", "Date", "Demand", "ma_7", "target"]].head(15)

,StockCode,Date,Demand,ma_7,target
0,10002,2010-12-29,0,NaN,0.0
1,10002,2010-12-30,0,NaN,0.0
2,10002,2010-12-31,0,NaN,0.0
3,10002,2011-01-01,0,NaN,0.0
4,10002,2011-01-02,0,NaN,0.0
5,10002,2011-01-03,0,NaN,0.0
6,10002,2011-01-04,0,0.000000,12.0
7,10002,2011-01-05,12,1.714286,60.0
8,10002,2011-01-06,60,10.285714,1.0
9,10002,2011-01-07,1,10.428571,0.0


In [14]:
validation_ma = baseline_data[
    (baseline_data["Date"] > "2011-08-19") & (baseline_data["Date"] <= "2011-10-12")
].copy()

validation_ma = validation_ma.dropna(subset=["ma_7", "target"])

In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae_ma7 = mean_absolute_error(validation_ma["target"], validation_ma["ma_7"])

rmse_ma7 = mean_squared_error(validation_ma["target"], validation_ma["ma_7"]) ** 0.5

print("7-Day Moving Average MAE:", mae_ma7)
print("7-Day Moving Average RMSE:", rmse_ma7)

7-Day Moving Average MAE: 9.553136212849502
7-Day Moving Average RMSE: 37.12059034442131


In [17]:
model_features = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_28",
    "day_of_week",
    "month",
    "week_of_year",
    "is_weekend",
]

In [18]:
from sklearn.linear_model import LinearRegression

X_train = train[model_features]
y_train = train["target"]

X_validation = validation[model_features]
y_validation = validation["target"]

In [19]:
linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](11,)","[-0.02, 0.01, 0.01,..., 0.3 ,-0.09, 3.78]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](11,)","['lag_1','lag_7','lag_14',...,'month','week_of_year','is_weekend']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,5.119
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,11
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,11


In [20]:
linear_validation_pred = linear_model.predict(X_validation)

In [21]:
comparison = pd.DataFrame(
    {"Actual": y_validation.values, "Predicted": linear_validation_pred}
)

comparison.head(10)

,Actual,Predicted
0,0.0,1.558807
1,0.0,0.386629
2,0.0,5.445637
3,25.0,3.957255
4,0.0,2.468872
5,0.0,2.719063
6,0.0,1.579398
7,0.0,2.223451
8,0.0,0.735069
9,0.0,5.794077


In [22]:
print("Min prediction:", linear_validation_pred.min())
print("Max prediction:", linear_validation_pred.max())
print("Negative predictions:", (linear_validation_pred < 0).sum())

Min prediction: -7.820641066170015
Max prediction: 256.7697919950221
Negative predictions: 14854


In [23]:
mae_linear = mean_absolute_error(y_validation, linear_validation_pred)

rmse_linear = mean_squared_error(y_validation, linear_validation_pred) ** 0.5

print("Linear Regression MAE:", mae_linear)
print("Linear Regression RMSE:", rmse_linear)

Linear Regression MAE: 8.798927561624463
Linear Regression RMSE: 35.23593393973345


In [24]:
coefficients = pd.DataFrame(
    {"Feature": model_features, "Coefficient": linear_model.coef_}
).sort_values("Coefficient", ascending=False)

coefficients

,Feature,Coefficient
10,is_weekend,3.781816
4,rolling_mean_7,0.681176
6,rolling_mean_28,0.539987
8,month,0.298514
1,lag_7,0.007967
2,lag_14,0.005147
0,lag_1,-0.015810
3,lag_28,-0.016340
9,week_of_year,-0.089469
5,rolling_std_7,-0.280860


In [26]:
import xgboost as xgb

print(xgb.__version__)

3.2.0


In [27]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
)

In [28]:
xgb_model.fit(X_train, y_train)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [29]:
xgb_validation_pred = xgb_model.predict(X_validation)

In [30]:
xgb_comparison = pd.DataFrame(
    {"Actual": y_validation.values, "Predicted": xgb_validation_pred}
)

xgb_comparison.head(10)

,Actual,Predicted
0,0.0,2.456470
1,0.0,3.223011
2,0.0,3.974154
3,25.0,3.977347
4,0.0,5.490889
5,0.0,5.102201
6,0.0,-0.103185
7,0.0,2.564731
8,0.0,4.480474
9,0.0,5.216669


In [31]:
print("Min prediction:", xgb_validation_pred.min())
print("Max prediction:", xgb_validation_pred.max())
print("Negative predictions:", (xgb_validation_pred < 0).sum())

Min prediction: -63.571396
Max prediction: 612.36365
Negative predictions: 4668


In [32]:
mae_xgb = mean_absolute_error(y_validation, xgb_validation_pred)

rmse_xgb = mean_squared_error(y_validation, xgb_validation_pred) ** 0.5

print("XGBoost MAE:", mae_xgb)
print("XGBoost RMSE:", rmse_xgb)

XGBoost MAE: 8.675399956501595
XGBoost RMSE: 35.58789610685837


In [33]:
xgb_results = pd.DataFrame(
    {"Actual": y_validation.values, "Predicted": xgb_validation_pred}
)

xgb_results["Error"] = xgb_results["Actual"] - xgb_results["Predicted"]

xgb_results["Absolute_Error"] = xgb_results["Error"].abs()

In [34]:
print(
    "Zero-demand MAE:",
    xgb_results.loc[xgb_results["Actual"] == 0, "Absolute_Error"].mean(),
)

print(
    "Non-zero demand MAE:",
    xgb_results.loc[xgb_results["Actual"] > 0, "Absolute_Error"].mean(),
)

Zero-demand MAE: 3.5264963060206465
Non-zero demand MAE: 18.31649177407204


In [35]:
high_demand = xgb_results[xgb_results["Actual"] >= 20]

print("High-demand observations:", len(high_demand))
print("High-demand MAE:", high_demand["Absolute_Error"].mean())
print("High-demand RMSE:", (high_demand["Error"] ** 2).mean() ** 0.5)

High-demand observations: 11782
High-demand MAE: 47.00839356266998
High-demand RMSE: 108.85909611036378


In [36]:
linear_results = pd.DataFrame(
    {"Actual": y_validation.values, "Predicted": linear_validation_pred}
)

linear_results["Error"] = linear_results["Actual"] - linear_results["Predicted"]

linear_results["Absolute_Error"] = linear_results["Error"].abs()

linear_high_demand = linear_results[linear_results["Actual"] >= 20]

print("High-demand observations:", len(linear_high_demand))
print("High-demand MAE:", linear_high_demand["Absolute_Error"].mean())
print("High-demand RMSE:", (linear_high_demand["Error"] ** 2).mean() ** 0.5)

High-demand observations: 11782
High-demand MAE: 46.16075144016594
High-demand RMSE: 108.76031881332261


In [37]:
zero_prediction = pd.Series(0, index=y_validation.index)

zero_mae = mean_absolute_error(y_validation, zero_prediction)

zero_rmse = mean_squared_error(y_validation, zero_prediction) ** 0.5

print("Zero Baseline MAE:", zero_mae)
print("Zero Baseline RMSE:", zero_rmse)

Zero Baseline MAE: 7.705673161194634
Zero Baseline RMSE: 37.840731405191534
